# Data Validation Analysis (Thesis Section 4.2)

This notebook reproduces the instrument validation analysis for the grading rubric applied to 2,550 student responses across 4 criteria (Clarity, Terminology, Coverage, Accuracy).

**Sections:**
1. Load & Explore Dataset
2. Score Distributions
3. Dimensionality Analysis (KMO, Bartlett, PCA, Parallel Analysis, EFA)
4. Reliability (Cronbach's alpha, McDonald's omega)
5. Bootstrap Confidence Intervals (10,000 resamples)
6. DIF Analysis (Block split + Difficulty terciles)
7. Multilevel Modeling (Mixed-effects, ICC)

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
from scipy import stats, optimize
from sklearn.decomposition import PCA
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid', font_scale=1.1)

BASE = '../Data Statsitical Analysis'
CRITERIA = ['Clarity_Score', 'Terminology_Score', 'Coverage_Score', 'Accuracy_Score']
CRITERIA_LABELS = ['Clarity', 'Terminology', 'Coverage', 'Accuracy']

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_excel(f'{BASE}/Data.xlsx')
X = df[CRITERIA].values
N, k = X.shape

# Derive question number and block
df['Q_num'] = df['ID'].str.extract(r'Q(\d+)')[0].astype(int)
df['Block'] = (df['Q_num'] > 42).astype(int)  # 0=Block A (Q1-42), 1=Block B (Q43-85)
df['Ability'] = df['Rubric_Grade']  # total score as ability proxy

print(f'Dataset: {N} responses, {k} criteria')
print(f'Questions: {df["Q_num"].nunique()} unique')
print(f'Block A: {(df["Block"]==0).sum()}, Block B: {(df["Block"]==1).sum()}')
df[CRITERIA].describe().round(3)

In [ ]:
# Correlation matrix
corr = df[CRITERIA].corr()
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CRITERIA_LABELS, yticklabels=CRITERIA_LABELS, ax=ax)
ax.set_title('Inter-Criteria Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, (crit, label) in enumerate(zip(CRITERIA, CRITERIA_LABELS)):
    ax = axes[i]
    max_score = 4 if 'Accuracy' in crit else 2
    bins = np.arange(-0.5, max_score + 1.5, 1)
    ax.hist(df[crit], bins=bins, color=colors[i], edgecolor='white', alpha=0.85, rwidth=0.8)
    ax.set_title(f'{label} (0-{max_score})', fontsize=12)
    ax.set_xlabel('Score')
    ax.set_ylabel('Frequency')
    ax.set_xticks(range(max_score + 1))
    # Add mean line
    ax.axvline(df[crit].mean(), color='black', linestyle='--', alpha=0.7, label=f'Mean={df[crit].mean():.2f}')
    ax.legend(fontsize=9)

fig.suptitle('Score Distributions by Rubric Criterion', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Dimensionality Analysis

In [ ]:
# ── Helper functions ──

def compute_kmo(X):
    """Kaiser-Meyer-Olkin measure of sampling adequacy."""
    k = X.shape[1]
    corr = np.corrcoef(X, rowvar=False)
    inv_corr = np.linalg.inv(corr)
    norms = np.sqrt(np.diag(inv_corr))
    partial = -inv_corr / np.outer(norms, norms)
    np.fill_diagonal(partial, 1.0)
    corr_sq_sum = np.sum(corr**2) - k
    partial_sq_sum = np.sum(partial**2) - k
    return corr_sq_sum / (corr_sq_sum + partial_sq_sum)


def bartlett_test(X):
    """Bartlett's test of sphericity."""
    n, p = X.shape
    corr = np.corrcoef(X, rowvar=False)
    det = np.linalg.det(corr)
    chi2 = -((n - 1) - (2 * p + 5) / 6) * np.log(det)
    dof = p * (p - 1) / 2
    p_val = 1 - stats.chi2.cdf(chi2, dof)
    return chi2, dof, p_val


def ml_factor_analysis_1factor(corr_mat, n_obs):
    """Maximum likelihood EFA with 1 factor."""
    p = corr_mat.shape[0]

    def neg_loglik(params):
        lam = params[:p].reshape(p, 1)
        psi = np.exp(params[p:])
        Sigma = lam @ lam.T + np.diag(psi)
        try:
            sign, logdet = np.linalg.slogdet(Sigma)
            if sign <= 0:
                return 1e10
            Sigma_inv = np.linalg.inv(Sigma)
            return 0.5 * (logdet + np.trace(Sigma_inv @ corr_mat))
        except np.linalg.LinAlgError:
            return 1e10

    eigvals, eigvecs = np.linalg.eigh(corr_mat)
    idx = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
    init_loadings = eigvecs[:, 0] * np.sqrt(max(eigvals[0], 0.01))
    init_uniquenesses = np.log(np.maximum(1.0 - init_loadings**2, 0.01))
    x0 = np.concatenate([init_loadings, init_uniquenesses])

    res = optimize.minimize(neg_loglik, x0, method='L-BFGS-B')
    lam = res.x[:p]
    psi = np.exp(res.x[p:])
    communalities = lam**2
    var_explained = np.sum(communalities) / p
    return lam, psi, communalities, var_explained

In [ ]:
# KMO and Bartlett's test
kmo = compute_kmo(X)
chi2, dof, p_val = bartlett_test(X)

print(f'KMO Measure of Sampling Adequacy: {kmo:.3f}')
print(f"Bartlett's Test: chi2 = {chi2:.1f}, df = {dof:.0f}, p < .001")

In [ ]:
# PCA eigenvalues
corr_mat = np.corrcoef(X, rowvar=False)
eigvals_corr = np.sort(np.linalg.eigvalsh(corr_mat))[::-1]
var_explained = eigvals_corr / eigvals_corr.sum() * 100
cumvar = np.cumsum(var_explained)

pca_table = pd.DataFrame({
    'Component': range(1, k+1),
    'Eigenvalue': eigvals_corr.round(3),
    '% Variance': var_explained.round(1),
    'Cumulative %': cumvar.round(1)
})
print('PCA on Correlation Matrix:')
pca_table

In [ ]:
# Parallel Analysis (Horn's method)
n_pa = 1000
pa_eigenvalues = np.zeros((n_pa, k))
for i in range(n_pa):
    random_data = np.random.normal(size=(N, k))
    random_corr = np.corrcoef(random_data, rowvar=False)
    pa_eigenvalues[i] = np.sort(np.linalg.eigvalsh(random_corr))[::-1]
pa_95 = np.percentile(pa_eigenvalues, 95, axis=0)

n_factors = np.sum(eigvals_corr > pa_95)

pa_table = pd.DataFrame({
    'Component': range(1, k+1),
    'Actual Eigenvalue': eigvals_corr.round(3),
    'PA 95th Percentile': pa_95.round(3),
    'Retain?': ['Yes' if eigvals_corr[i] > pa_95[i] else 'No' for i in range(k)]
})
print(f'Factors retained by parallel analysis: {n_factors}')
pa_table

In [ ]:
# Scree plot with parallel analysis
fig, ax = plt.subplots(figsize=(7, 4.5))
components = np.arange(1, k + 1)
ax.plot(components, eigvals_corr, 'bo-', linewidth=2, markersize=8, label='Actual Eigenvalues')
ax.plot(components, pa_95, 'r^--', linewidth=2, markersize=8, label='Parallel Analysis (95th %ile)')
ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.7, label='Kaiser Criterion')
ax.set_xlabel('Component', fontsize=12)
ax.set_ylabel('Eigenvalue', fontsize=12)
ax.set_title('Scree Plot with Parallel Analysis', fontsize=13)
ax.set_xticks(components)
ax.legend(fontsize=10)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
# EFA with 1 factor (Maximum Likelihood)
lam_efa, psi_efa, communalities_efa, var_expl_efa = ml_factor_analysis_1factor(corr_mat, N)

efa_table = pd.DataFrame({
    'Criterion': CRITERIA_LABELS,
    'Loading': lam_efa.round(3),
    'Communality': communalities_efa.round(3)
})
print(f'EFA (1 Factor, ML) — Variance explained: {var_expl_efa*100:.1f}%')
efa_table

## 4. Reliability

In [ ]:
def compute_alpha(data):
    n_items = data.shape[1]
    item_vars = np.var(data, axis=0, ddof=1)
    total_var = np.var(np.sum(data, axis=1), ddof=1)
    return (n_items / (n_items - 1)) * (1 - np.sum(item_vars) / total_var)


def compute_omega_from_loadings(lam, psi):
    sum_lam = np.sum(lam)
    return sum_lam**2 / (sum_lam**2 + np.sum(psi))


alpha_val = compute_alpha(X)
omega_total = compute_omega_from_loadings(lam_efa, psi_efa)

print(f"Cronbach's alpha:   {alpha_val:.3f}")
print(f"McDonald's omega_t: {omega_total:.3f}")
print(f"Difference:         {abs(omega_total - alpha_val):.3f}")
print(f"Both indicate {'excellent' if min(alpha_val, omega_total) >= 0.9 else 'good'} reliability")

## 5. Bootstrap Confidence Intervals (10,000 resamples)

In [ ]:
def compute_omega(data):
    corr = np.corrcoef(data, rowvar=False)
    lam, psi, _, _ = ml_factor_analysis_1factor(corr, data.shape[0])
    return compute_omega_from_loadings(lam, psi), lam, psi


def compute_item_total_spearman(data):
    total = np.sum(data, axis=1)
    return np.array([stats.spearmanr(data[:, j], total)[0] for j in range(data.shape[1])])


N_BOOTSTRAP = 10_000
boot_alphas = np.zeros(N_BOOTSTRAP)
boot_omegas = np.zeros(N_BOOTSTRAP)
boot_item_totals = np.zeros((N_BOOTSTRAP, k))

print(f'Running {N_BOOTSTRAP:,} bootstrap iterations...')
for b in range(N_BOOTSTRAP):
    if (b + 1) % 2500 == 0:
        print(f'  {b+1:,} done')
    idx = np.random.choice(N, size=N, replace=True)
    X_boot = X[idx]
    boot_alphas[b] = compute_alpha(X_boot)
    boot_item_totals[b] = compute_item_total_spearman(X_boot)
    try:
        omega_b, _, _ = compute_omega(X_boot)
        boot_omegas[b] = omega_b
    except Exception:
        boot_omegas[b] = np.nan

valid_omega = boot_omegas[~np.isnan(boot_omegas)]
print(f'Valid omega bootstraps: {len(valid_omega):,}/{N_BOOTSTRAP:,}')

In [ ]:
# Confidence intervals
alpha_ci = np.percentile(boot_alphas, [2.5, 97.5])
omega_ci = np.percentile(valid_omega, [2.5, 97.5])
item_total_cis = np.percentile(boot_item_totals, [2.5, 97.5], axis=0)
point_item_totals = compute_item_total_spearman(X)

print(f"Cronbach's alpha:  {alpha_val:.3f}  95% CI [{alpha_ci[0]:.3f}, {alpha_ci[1]:.3f}]")
print(f"McDonald's omega:  {omega_total:.3f}  95% CI [{omega_ci[0]:.3f}, {omega_ci[1]:.3f}]")
print()
print('Item-Total Correlations (Spearman) with 95% CIs:')
ci_table = pd.DataFrame({
    'Criterion': CRITERIA_LABELS,
    'r_s': point_item_totals.round(3),
    'CI Lower': item_total_cis[0].round(3),
    'CI Upper': item_total_cis[1].round(3)
})
ci_table

In [ ]:
# Forest plot of reliability CIs
fig, ax = plt.subplots(figsize=(8, 4))
measures = ["Cronbach's $\\alpha$", "McDonald's $\\omega_t$"]
estimates = [alpha_val, omega_total]
lowers = [alpha_ci[0], omega_ci[0]]
uppers = [alpha_ci[1], omega_ci[1]]

y_pos = np.arange(len(measures))
xerr = np.array([[e - l, u - e] for e, l, u in zip(estimates, lowers, uppers)]).T

ax.errorbar(estimates, y_pos, xerr=xerr, fmt='o', color='#2c3e50',
            capsize=6, capthick=2, linewidth=2, markersize=10)
ax.set_yticks(y_pos)
ax.set_yticklabels(measures, fontsize=13)
ax.set_xlabel('Reliability Coefficient', fontsize=12)
ax.set_title('Reliability Estimates with 95% Bootstrap CIs', fontsize=14, pad=15)
ax.axvline(x=0.90, color='green', linestyle='--', alpha=0.5, label='Excellent threshold (0.90)')
ax.legend(fontsize=10, loc='upper left')

for i, (est, lo, hi) in enumerate(zip(estimates, lowers, uppers)):
    ax.annotate(f'{est:.3f} [{lo:.3f}, {hi:.3f}]',
                xy=(hi, i), xytext=(8, 0),
                textcoords='offset points', fontsize=10, va='center', ha='left')

ax.set_xlim(0.88, 0.97)
ax.set_ylim(-0.5, 1.5)
plt.tight_layout()
plt.show()

## 6. DIF Analysis

In [ ]:
def compute_dif_metrics(data, group_col, group_vals, ability_col='Ability'):
    """Compute SMD, partial r, and MH classification for DIF."""
    results = []
    for criterion, label in zip(CRITERIA, CRITERIA_LABELS):
        g1 = data[data[group_col] == group_vals[0]]
        g2 = data[data[group_col] == group_vals[1]]

        # Raw SMD
        m1, m2 = g1[criterion].mean(), g2[criterion].mean()
        s1, s2 = g1[criterion].std(), g2[criterion].std()
        n1, n2 = len(g1), len(g2)
        sd_pool = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
        smd = (m2 - m1) / sd_pool if sd_pool > 0 else 0

        # Partial correlation (controlling for ability)
        combined = pd.concat([g1, g2])
        group_indicator = (combined[group_col] == group_vals[1]).astype(int)
        ability = combined[ability_col].values
        criterion_vals = combined[criterion].values
        group_vals_numeric = group_indicator.values

        slope_c, intercept_c = np.polyfit(ability, criterion_vals, 1)
        resid_c = criterion_vals - (slope_c * ability + intercept_c)
        slope_g, intercept_g = np.polyfit(ability, group_vals_numeric, 1)
        resid_g = group_vals_numeric - (slope_g * ability + intercept_g)
        partial_r = np.corrcoef(resid_c, resid_g)[0, 1]

        # MH classification
        abs_pr = abs(partial_r)
        mh_class = 'A' if abs_pr < 0.10 else ('B' if abs_pr < 0.25 else 'C')

        # SMD interpretation
        abs_smd = abs(smd)
        smd_interp = 'Negligible' if abs_smd < 0.25 else ('Small' if abs_smd < 0.50 else ('Medium' if abs_smd < 0.80 else 'Large'))

        results.append({
            'Criterion': label, 'Raw_SMD': round(smd, 3),
            'SMD_Interpretation': smd_interp, 'Partial_r': round(partial_r, 3),
            'MH_Class': mh_class,
            'DIF_Status': 'Negligible' if mh_class == 'A' else ('Moderate' if mh_class == 'B' else 'Large')
        })
    return pd.DataFrame(results)

In [ ]:
# Block split DIF
block_dif = compute_dif_metrics(df, 'Block', [0, 1])
print('DIF Results — Block Split (Q1-42 vs Q43-85):')
block_dif

In [ ]:
# Difficulty tercile DIF
q_means = df.groupby('Q_num')['Rubric_Grade'].mean()
tercile_cuts = q_means.quantile([1/3, 2/3])

def assign_tercile(q):
    m = q_means[q]
    if m < tercile_cuts.iloc[0]: return 'Hard'
    elif m < tercile_cuts.iloc[1]: return 'Medium'
    else: return 'Easy'

df['Difficulty'] = df['Q_num'].apply(assign_tercile)

print(f'Tercile boundaries: Hard < {tercile_cuts.iloc[0]:.2f}, Medium < {tercile_cuts.iloc[1]:.2f}, Easy >= {tercile_cuts.iloc[1]:.2f}')
for g in ['Easy', 'Medium', 'Hard']:
    print(f'  {g}: {(df["Difficulty"] == g).sum()} responses')

df_easy_hard = df[df['Difficulty'].isin(['Easy', 'Hard'])].copy()
diff_dif = compute_dif_metrics(df_easy_hard, 'Difficulty', ['Easy', 'Hard'])
print()
print('DIF Results — Difficulty Tercile (Easy vs Hard):')
diff_dif

In [ ]:
# Side-by-side DIF comparison figure
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
y_pos = np.arange(len(CRITERIA_LABELS))
bar_w = 0.35

for ax, dif_df, title in [(axes[0], block_dif, '(a) Block Split (Q1-42 vs Q43-85)'),
                           (axes[1], diff_dif, '(b) Difficulty Tercile (Easy vs Hard)')]:
    raw_smds = dif_df['Raw_SMD'].values
    partial_rs = dif_df['Partial_r'].values
    ax.barh(y_pos - bar_w/2, np.abs(raw_smds), bar_w, label='|Raw SMD|', color='#3498db')
    ax.barh(y_pos + bar_w/2, np.abs(partial_rs), bar_w, label='|Partial r|', color='#e74c3c')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(CRITERIA_LABELS, fontsize=11)
    ax.set_xlabel('Effect Size', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.axvline(x=0.25, color='gray', linestyle='--', alpha=0.5, label='SMD threshold')
    ax.axvline(x=0.10, color='orange', linestyle=':', alpha=0.5, label='Partial r threshold')
    ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

## 7. Multilevel Modeling

In [ ]:
mlm_results = []

for criterion, label in zip(CRITERIA, CRITERIA_LABELS):
    model_df = df[['Q_num', 'Ability', 'Block', criterion]].copy()
    model_df = model_df.rename(columns={criterion: 'Score'})
    model = MixedLM.from_formula('Score ~ Ability + Block', groups='Q_num', data=model_df)
    result = model.fit(reml=True)

    resid_var = result.scale
    q_var = float(result.cov_re.iloc[0, 0])
    icc = q_var / (q_var + resid_var)

    mlm_results.append({
        'Criterion': label,
        'Block_Effect': round(result.fe_params['Block'], 4),
        'Block_SE': round(result.bse_fe['Block'], 4),
        'Block_p': round(result.pvalues['Block'], 4),
        'Ability_Effect': round(result.fe_params['Ability'], 4),
        'Ability_SE': round(result.bse_fe['Ability'], 4),
        'ICC': round(icc, 3),
        'Var_Question': round(q_var, 4),
        'Var_Residual': round(resid_var, 4)
    })

mlm_df = pd.DataFrame(mlm_results)
print('Multilevel Model: Score ~ Ability + Block + (1|Question)')
mlm_df

In [ ]:
# Summary: ICC values
print('ICC values (proportion of variance due to question-level differences):')
for _, row in mlm_df.iterrows():
    print(f"  {row['Criterion']:<15} ICC = {row['ICC']:.3f}")
print()
print('Interpretation: Low ICCs indicate rubric scoring is consistent across questions.')
print('Block effect significance:')
for _, row in mlm_df.iterrows():
    sig = 'significant' if row['Block_p'] < 0.05 else 'not significant'
    print(f"  {row['Criterion']:<15} Block effect = {row['Block_Effect']:+.4f}, p = {row['Block_p']:.4f} ({sig})")

In [ ]:
# Cross-validate against pre-saved results
saved = pd.read_csv(f'{BASE}/bootstrap_ci_results.csv')
print('Pre-saved bootstrap CI results (for comparison):')
saved